In [3]:
import torch
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer
from read_jsonl import read_jsonl
import pandas as pd
from captum.attr import LayerIntegratedGradients

In [4]:
names = ["bert", "distilbert"]
tokenizer_distil = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

model_distil = DistilBertForSequenceClassification.from_pretrained("./results/distilbert/checkpoint-37")
model_distil.eval()

# Detect MPS (Apple Silicon GPU)
force_cpu = False
device = torch.device("mps" if torch.backends.mps.is_available() and not force_cpu else "cpu")
print(f"Using device: {device}")
model_distil.to(device)

Using device: cpu


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [5]:
eval_df = read_jsonl("DB-bio/combined_val_and_val_sft_anonymized.jsonl")
len(eval_df)

486

In [6]:
def forward(text, logit = None):
    
    inputs = tokenizer_distil(text,return_tensors="pt", truncation=True, padding='max_length', max_length=128)
    ids = inputs["input_ids"]
    logits = model_distil(input_ids=ids, attention_mask=inputs["attention_mask"]).logits
    if logit is not None:
        return torch.softmax(logits, dim=1)[0][logit].item()
    return torch.argmax(logits).item()

In [7]:
def forward_captum(_input_ids, _attention_mask, _model,):
    out = _model(input_ids=_input_ids, attention_mask=_attention_mask).logits
    return out

lig_distilbert = LayerIntegratedGradients(forward_captum, model_distil.distilbert.embeddings)

def compute_attributions(_text):
    _inputs = tokenizer_distil(_text, return_tensors="pt", truncation=True, padding='max_length', max_length=128)
    _inputs.to(device)
    _input_ids = _inputs['input_ids']
    _input_ids.to(device)
    _attention_mask = _inputs['attention_mask']
    _attention_mask.to(device)
    baseline = torch.zeros_like(_input_ids)
    baseline.to(device)
    
    with torch.no_grad():
        _logits = model_distil(**_inputs).logits
        target = torch.argmax(_logits, dim=1).item()
        conf = torch.softmax(_logits, dim=1)[0][target].item()
    
    _attributions, _delta = lig_distilbert.attribute(
        inputs=_input_ids,
        baselines=baseline,
        additional_forward_args=(_attention_mask,model_distil),
        target=target,
        return_convergence_delta=True
    )
    
    scores = _attributions.sum(dim=-1).squeeze(0)
    tokens = tokenizer_distil.convert_ids_to_tokens(_input_ids[0])
    filtered = [(_input_ids[0][i].item(), tokens[i], scores[i]) for i in range(len(tokens)) if tokens[i] not in ["[CLS]", "[SEP]", "[PAD]", '.', ',', "(", ")"]]
    return filtered

In [8]:
compute_attributions(text)

NameError: name 'text' is not defined

In [85]:
df_rm = pd.read_csv("DB-bio/global_analysis_removed_tokens_top_1K.csv")
df_add = pd.read_csv("DB-bio/global_analysis_added_tokens_top_1K.csv")
removed_tokens = list(df_rm["token_id"])
added_tokens = list(df_add["token_id"])

In [7]:
embedding_matrix = model_distil.get_input_embeddings().weight
embedding_matrix.shape

torch.Size([30522, 768])

In [8]:
embeddings_removed = embedding_matrix[removed_tokens]
embeddings_added = embedding_matrix[added_tokens]
embeddings_removed.shape, embeddings_added.shape

(torch.Size([1000, 768]), torch.Size([1000, 768]))

In [9]:
embeddings_removed = torch.nn.functional.normalize(embeddings_removed, dim=1)
embeddings_added = torch.nn.functional.normalize(embeddings_added, dim=1)
similarity_matrix = torch.matmul(embeddings_removed, embeddings_added.T)

In [15]:
most_similar_for_removed = similarity_matrix.argmax(dim=1)
most_similar_for_added = similarity_matrix.argmax(dim=0)

int

In [18]:
removed_tokens_replacement = {token: added_tokens[most_similar_for_removed[i]] for i,token in enumerate(removed_tokens)}
added_tokens_replacement = {token: removed_tokens[most_similar_for_added[i]] for i,token in enumerate(added_tokens)}

In [39]:
removed_tokens_replacement[removed_tokens[371]]

2885

In [93]:
def get_candidates(target):
    importance_scores = compute_attributions(target)

    aggregated = {}
    for i,t, s in importance_scores:
        _, prev = aggregated.get(i, (t,s))
        aggregated[i] = (t, (prev + s))
    
    ordered = sorted(aggregated.items(), key=lambda x: x[1][1], reverse=True)
    return [i for i, (t, s) in ordered if s > 0]

In [ ]:
def attack2(target):
    print(target + '\n')
    original_pred = forward(target)

    removed_tokens_in_attack = []
    replacements = set()
    for i in range(5):
        cands = get_candidates(target)
        adversarial_ids = tokenizer_distil(target)["input_ids"][1:-1]
        
        for token in cands[:5]:
            removed_tokens_in_attack.append(token)
            adversarial_ids_tmp = []
            for i in adversarial_ids:
                if i == token:
                    if i in removed_tokens_replacement:
                        replacements.add(removed_tokens_replacement[i])
                        adversarial_ids_tmp.append(removed_tokens_replacement[i])
                else:
                    adversarial_ids_tmp.append(i)
            
            # evaluate model
            target = tokenizer_distil.decode(adversarial_ids_tmp)
            pred = forward(target)
            if pred != original_pred:
                removed_tokens_in_attack = [tokenizer_distil.decode(token) for token in removed_tokens_in_attack]
                print("Attack complete: \nRemoved: {}\nReplacements: {}\n".format(removed_tokens_in_attack, replacements), target)
                return
            adversarial_ids = adversarial_ids_tmp
            
    print("Attack failed: \n",target)

In [119]:
def attack3(target):
    print(target + '\n')
    original_pred = forward(target)
    next_path = [(tokenizer_distil(target)["input_ids"][1:-1], 0, 1)]
    visited = set()
    found = []
    best_depth = 1000000
    step = 0
    
    while len(next_path) > 0:
        if step > 10:
            return min(found, key=lambda x: x[1])
        step += 1
        
        path, depth, score = next_path.pop(0)
        print(depth)
        if depth >= best_depth:
            continue
        for important_token in get_candidates(tokenizer_distil.decode(path))[:5]:
            new_path = []
            for id in path:
                if id == important_token:
                    if id in removed_tokens_replacement:
                        new_path.append(removed_tokens_replacement[id])
                else:
                    new_path.append(id)
                    
            if tuple(new_path) in visited:
                continue
            else:
                visited.add(tuple(new_path))     
                
            new_path_text = tokenizer_distil.decode(new_path)
            score = forward(new_path_text, original_pred)
            print("Depth=[{}] - score: [{}] - token: [{}]".format(depth, score, important_token))
            
            if score <= 0.5:
                best_depth = depth
                found.append((new_path_text, depth))
                print("Attack succesful [depth={}]: \n".format(depth), new_path_text)
                continue
            
            if depth + 1 < best_depth:
                next_path.append((new_path, depth + 1, score))
            
        next_path = sorted(next_path, key=lambda x: x[2])

In [120]:
attack3(text)

Stephen J. Gordon (born 4 September 1986) is a chess grandmaster from Oldham, Greater Manchester, England. In September 2004 he took a break from his A-level studies at The Blue Coat School, Oldham to compete in the thirteenth Monarch Assurance Isle of Man International. In 2005, while still a FIDE Master, he finished 6th in the British Championships ahead of a Grandmaster and several International Masters. At the EU Individual Open Chess Championship held at Liverpool in 2006, he led the tournament after eight rounds and finished a very creditable (joint) second, a half point behind winner Nigel Short and level with Luke McShane among others. Probably his best result to date however, was second place in the 2007 British Championship, narrowly losing his share of the lead in the final round. In previous rounds, he defeated both tournament victor Jacob Aagaard and previous champion Jonathan Rowson. By 2008, his rating had reached grandmaster level, although the title itself had not yet 

KeyboardInterrupt: 

In [20]:
def attack(target):
    print(target)
    original_pred = forward(target)
    original_ids = tokenizer_distil(target)["input_ids"][1:-1]
    print(tokenizer_distil.decode(original_ids))
    print(len(original_ids), original_pred)
    
    target2 = original_ids.copy()
    rem_tokens_by_attack = []
    while True:
        compute_attributions
        cands = []
        for token in removed_tokens:
            if token in target2:
                target3 = [t if t != token else removed_tokens_replacement[token] for t in target2]
                target_text = tokenizer_distil.decode(target3)
                pred = forward(target_text, original_pred)
                cands.append((token, pred))
        
        if len(cands) == 0:
            print("Attack was not successful!")
        cands = sorted(cands,key=lambda x: x[1])
        removed_token = cands[0][0]
        print("removed_token: ", removed_token)
        rem_tokens_by_attack.append(removed_token)
        target2 = [t if t != removed_token else removed_tokens_replacement[removed_token] for t in target2 ]
        target_text = tokenizer_distil.decode(target2)
        pred = forward(target_text)
        if pred != original_pred:
            important_tokens = [tokenizer_distil.decode([t]) for t in rem_tokens_by_attack] 
            print("attack successful w/ {} tokens\nremoved tokens:{}\n\nText after attack: \n".format(len(target2), important_tokens), target_text)
            break

In [21]:
for text in eval_df["text"]:
    attack(text)
    print("----------------------------------------------------------------------------------------------------------------------------------------")

Stephen J. Gordon (born 4 September 1986) is a chess grandmaster from Oldham, Greater Manchester, England. In September 2004 he took a break from his A-level studies at The Blue Coat School, Oldham to compete in the thirteenth Monarch Assurance Isle of Man International. In 2005, while still a FIDE Master, he finished 6th in the British Championships ahead of a Grandmaster and several International Masters. At the EU Individual Open Chess Championship held at Liverpool in 2006, he led the tournament after eight rounds and finished a very creditable (joint) second, a half point behind winner Nigel Short and level with Luke McShane among others. Probably his best result to date however, was second place in the 2007 British Championship, narrowly losing his share of the lead in the final round. In previous rounds, he defeated both tournament victor Jacob Aagaard and previous champion Jonathan Rowson. By 2008, his rating had reached grandmaster level, although the title itself had not yet 

KeyboardInterrupt: 

In [45]:
text = "Stephen J. Gordon (born 4 September 1986) is a chess grandmaster from Oldham, Greater Manchester, England. In September 2004 he took a break from his A-level studies at The Blue Coat School, Oldham to compete in the thirteenth Monarch Assurance Isle of Man International. In 2005, while still a FIDE Master, he finished 6th in the British Championships ahead of a Grandmaster and several International Masters. At the EU Individual Open Chess Championship held at Liverpool in 2006, he led the tournament after eight rounds and finished a very creditable (joint) second, a half point behind winner Nigel Short and level with Luke McShane among others. Probably his best result to date however, was second place in the 2007 British Championship, narrowly losing his share of the lead in the final round. In previous rounds, he defeated both tournament victor Jacob Aagaard and previous champion Jonathan Rowson. By 2008, his rating had reached grandmaster level, although the title itself had not yet been secured. At the British Championship in Liverpool, he almost repeated his performance of the previous year, by taking a share of third place. He was the British under-21 Champion each consecutive year between 2005 and 2008. He became a grandmaster on 1 August 2009. He has been one of the co-presenters of the chess podcast The Full English Breakfast since its inaugural show in October 2010."
forward(text)

0

In [41]:
text = "Person is a chess grandmaster from europe. they took a break from their A-level studies at The Blue Coat School, Oldham to compete in the thirteenth Monarch Assurance Isle of Man International. while still a FIDE Master, they finished in the uk Championships ahead of a Grandmaster and several International Masters. At the EU Individual Open Chess Championship held at Liverpool, they led the tournament after eight rounds and finished a very creditable joint second, a half point behind winner and level with Person among others. Probably their best result to date however, was second place in the uk Championship, narrowly losing their share of the lead in the final round. In previous rounds, they defeated both tournament victor Person and previous champion Person. By 2008, their rating had reached grandmaster level, although the title itself had not yet been secured. At the uk Championship in Liverpool, they almost repeated their performance of the previous year, by taking a share of third place. They was the uk under-21 Champion each consecutive year between. They became a grandmaster. They has been one of the co-presenters of the chess podcast The Full european Breakfast since its inaugural show."
forward(text)

1